# Cell 0
- Producer (S3 JSONL) -> Bronze Delta Table
- Uses Boto3 to bypass Serverless Spark S3 auth issues
- Reads JSONL.gz files -> Spark DataFrame -> Delta Table
- Source: s3://enterprise-raw-lakehouse/crypto/ohlc_1m/
- Target: enterprise.bronze_market.crypto_ohlc_1m
- Mode: Autoloader (CloudFiles) for streaming ingestion
- Automatically checks the target Delta table to skip already-ingested files.
- `batch`: Runs once and processes all new files.
- `realtime`: Runs continuously, polling S3 every 60 seconds for new data.

# Cell 1: Library Imports

In [0]:
import gzip
import json
import time
from datetime import datetime

import boto3
from pyspark.sql.functions import col, current_timestamp, to_timestamp
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)

# Cell 2: Configuration & Widgets

In [0]:
# Widgets for parameterization
dbutils.widgets.text("s3_bucket", "enterprise-raw-lakehouse")
dbutils.widgets.text("s3_prefix", "crypto/ohlc_1m")
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("schema", "bronze_ingestion")
dbutils.widgets.text("table_name", "brz_crypto_ohlc_1m")
dbutils.widgets.text("aws_access_key", "", "AWS Access Key")
dbutils.widgets.text("aws_secret_key", "", "AWS Secret Key")
dbutils.widgets.text("aws_region", "eu-central-1", "AWS Region")
dbutils.widgets.text("mode", "batch")  # 'batch' or 'realtime'

# Parse parameters
BUCKET = dbutils.widgets.get("s3_bucket").strip()
PREFIX = dbutils.widgets.get("s3_prefix").strip().strip("/")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
TABLE_NAME = dbutils.widgets.get("table_name").strip()
ACCESS_KEY = dbutils.widgets.get("aws_access_key").strip()
SECRET_KEY = dbutils.widgets.get("aws_secret_key").strip()
REGION = dbutils.widgets.get("aws_region").strip()
MODE = dbutils.widgets.get("mode").strip().lower()

TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

print(f"🚀 Source: s3://{BUCKET}/{PREFIX}/")
print(f"🎯 Target: {TARGET_TABLE}")
print(f"⚙️ Mode:   {MODE.upper()}")

# Cell 3: Initialize S3 Client

In [0]:
if not ACCESS_KEY or not SECRET_KEY:
    raise ValueError("❌ Missing AWS Keys! Please provide aws_access_key and aws_secret_key.")

s3 = boto3.client(
    "s3", aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY, region_name=REGION
)
print("✅ S3 Client Initialized.")

# Cell 4: Scan S3 files -> Deduplication -> Write to Delta  

In [0]:
%skip


def list_jsonl_files(bucket, prefix):
    """List all .jsonl.gz files in the bucket/prefix"""
    files = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                key = obj["Key"]
                if key.endswith(".jsonl.gz"):
                    files.append(key)
    return files


def run_batch():
    start_time = datetime.now()
    print(f"\n--- ⏳ Starting Batch at {start_time} ---")

    # --- Step 1: List Files ---
    print(f"🔍 Listing files in {PREFIX}...")
    all_file_keys = list_jsonl_files(BUCKET, PREFIX)
    print(f"📄 Found total {len(all_file_keys)} files in S3.")

    # --- Step 2: Deduplication (Checkpointing) ---
    processed_files = set()
    try:
        if spark.catalog.tableExists(TARGET_TABLE):
            # Query existing files in the table to avoid re-processing
            df_existing = spark.sql(f"SELECT DISTINCT source_file FROM {TARGET_TABLE}")
            rows = df_existing.collect()
            processed_files = {r["source_file"] for r in rows if r["source_file"]}
            print(f"📚 Found {len(processed_files)} already processed files in Delta table.")
    except Exception as e:
        print(f"⚠️ Warning: Could not query existing table: {e}. Proceeding with all files.")

    # Filter out already processed files
    file_keys = [k for k in all_file_keys if k not in processed_files]
    skipped_count = len(all_file_keys) - len(file_keys)

    if skipped_count > 0:
        print(f"⏭️ Skipped {skipped_count} files (already ingested).")

    if not file_keys:
        print("✅ No new files to process.")
        return

    print(f"🚀 Files to process: {len(file_keys)}")

    # --- Step 3: Process Files (Download & Parse) ---
    all_rows = []
    error_count = 0

    # Batch limit to prevent OOM in continuous mode
    MAX_FILES_PER_BATCH = 500000
    files_to_process = file_keys[:MAX_FILES_PER_BATCH]

    if len(file_keys) > MAX_FILES_PER_BATCH:
        print(
            f"⚠️ Batch Limit: processing first {MAX_FILES_PER_BATCH} of {len(file_keys)} new files."
        )

    for key in files_to_process:
        try:
            print(f"⬇️ Reading: {key}")
            obj = s3.get_object(Bucket=BUCKET, Key=key)
            with gzip.GzipFile(fileobj=obj["Body"]) as gz:
                for line in gz:
                    if line.strip():
                        record = json.loads(line)
                        record["source_file"] = key  # Add metadata for deduplication
                        all_rows.append(record)
        except Exception as e:
            print(f"❌ Error reading {key}: {e}")
            error_count += 1

    print(f"✅ Extracted {len(all_rows)} records. Errors: {error_count}")

    # --- Step 4: Write to Delta ---
    if all_rows:
        # Define Schema explicitly
        schema = StructType(
            [
                StructField("source", StringType(), True),
                StructField("symbol", StringType(), True),
                StructField("interval", StringType(), True),
                StructField("event_ts", LongType(), True),
                StructField("event_time", StringType(), True),
                StructField("low", DoubleType(), True),
                StructField("high", DoubleType(), True),
                StructField("open", DoubleType(), True),
                StructField("close", DoubleType(), True),
                StructField("volume", DoubleType(), True),
                StructField("run_id", StringType(), True),
                StructField("ingest_ts", StringType(), True),
                StructField("window_start", StringType(), True),
                StructField("window_end", StringType(), True),
                StructField("_producer_run_id", StringType(), True),
                StructField("_produced_at", StringType(), True),
                StructField("_merge_minutes", LongType(), True),
                StructField("source_file", StringType(), True),
            ]
        )

        df = spark.createDataFrame(all_rows, schema=schema)

        # Transformations & Timestamp Casting
        df_final = (
            df.withColumn("event_time", to_timestamp(col("event_time")))
            .withColumn("ingestion_ts", to_timestamp(col("ingest_ts")))
            .withColumn("bronze_proc_ts", current_timestamp())
        )

        spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA}")

        (
            df_final.write.format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(TARGET_TABLE)
        )
        print(f"💾 Written {len(all_rows)} rows to {TARGET_TABLE}")
    else:
        print("⚠️ No valid rows extracted.")

In [0]:
def list_jsonl_files(bucket, prefix):
    """List all .jsonl.gz files in the bucket/prefix"""
    files = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                key = obj["Key"]
                if key.endswith(".jsonl.gz"):
                    files.append(key)
    return files


def run_batch():
    start_time = datetime.now()
    print(f"\n--- ⏳ Starting Batch at {start_time} ---")

    # --- Step 1: List Files ---
    print(f"🔍 Listing files in {PREFIX}...")
    all_file_keys = list_jsonl_files(BUCKET, PREFIX)
    print(f"📄 Found total {len(all_file_keys)} files in S3.")

    # --- Step 2: Deduplication (Checkpointing) ---
    processed_files = set()
    try:
        if spark.catalog.tableExists(TARGET_TABLE):
            # Query existing files in the table to avoid re-processing
            df_existing = spark.sql(f"SELECT DISTINCT source_file FROM {TARGET_TABLE}")
            rows = df_existing.collect()
            processed_files = {r["source_file"] for r in rows if r["source_file"]}
            print(f"📚 Found {len(processed_files)} already processed files in Delta table.")
    except Exception as e:
        print(f"⚠️ Warning: Could not query existing table: {e}. Proceeding with all files.")

    # Filter out already processed files
    file_keys = [k for k in all_file_keys if k not in processed_files]
    skipped_count = len(all_file_keys) - len(file_keys)

    if skipped_count > 0:
        print(f"⏭️ Skipped {skipped_count} files (already ingested).")

    # --- Step 3: Process in Chunks ---
    files_to_process = file_keys
    CHUNK_SIZE = 500  # Adjust based on memory (1000 files is safe for free tier)
    total_files = len(files_to_process)

    if total_files == 0:
        print("✅ No new files to process.")
        return

    print(f"🚀 Total files to process: {total_files}. Processing in chunks of {CHUNK_SIZE}...")

    # Iterate in chunks
    for i in range(0, total_files, CHUNK_SIZE):
        chunk_files = files_to_process[i : i + CHUNK_SIZE]
        all_rows = []
        error_count = 0

        print(f"\n--- Batch {i // CHUNK_SIZE + 1} / {(total_files // CHUNK_SIZE) + 1} ---")
        print(f"📥 Reading files {i} to {min(i + CHUNK_SIZE, total_files)}...")

        for key in chunk_files:
            try:
                # print(f"⬇️ Reading: {key}") # Optional: Comment out to reduce log noise
                obj = s3.get_object(Bucket=BUCKET, Key=key)
                with gzip.GzipFile(fileobj=obj["Body"]) as gz:
                    for line in gz:
                        if line.strip():
                            record = json.loads(line)
                            record["source_file"] = key  # Add metadata for deduplication
                            all_rows.append(record)
            except Exception as e:
                print(f"❌ Error reading {key}: {e}")
                error_count += 1

        print(f"✅ Extracted {len(all_rows)} records in this chunk. Errors: {error_count}")

        # --- Step 4: Write Chunk to Delta ---
        if all_rows:
            # Define Schema explicitly
            schema = StructType(
                [
                    StructField("source", StringType(), True),
                    StructField("symbol", StringType(), True),
                    StructField("interval", StringType(), True),
                    StructField("event_ts", LongType(), True),
                    StructField("event_time", StringType(), True),
                    StructField("low", DoubleType(), True),
                    StructField("high", DoubleType(), True),
                    StructField("open", DoubleType(), True),
                    StructField("close", DoubleType(), True),
                    StructField("volume", DoubleType(), True),
                    StructField("run_id", StringType(), True),
                    StructField("ingest_ts", StringType(), True),
                    StructField("window_start", StringType(), True),
                    StructField("window_end", StringType(), True),
                    StructField("_producer_run_id", StringType(), True),
                    StructField("_produced_at", StringType(), True),
                    StructField("_merge_minutes", LongType(), True),
                    StructField("source_file", StringType(), True),
                ]
            )

            df = spark.createDataFrame(all_rows, schema=schema)

            # Transformations & Timestamp Casting
            df_final = (
                df.withColumn("event_time", to_timestamp(col("event_time")))
                .withColumn("ingestion_ts", to_timestamp(col("ingest_ts")))
                .withColumn("bronze_proc_ts", current_timestamp())
            )

            # Create Database if not exists (only need to run once, but safe here)
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA}")

            (
                df_final.write.format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(TARGET_TABLE)
            )
            print(f"💾 Written chunk to {TARGET_TABLE}")

            # Clear memory
            del all_rows
            del df
            del df_final
        else:
            print("⚠️ No valid rows extracted in this chunk.")

# Cell 5: Execution Controlle

In [0]:
try:
    MODE = dbutils.widgets.get("mode").strip().lower()
except Exception:
    MODE = "batch"  # Default if widget fails

# Safety check for imports/setup
if "boto3" not in globals() or "s3" not in globals():
    print(
        "❌ Error: Setup cells not run! Please run the 'Imports' and 'Configuration' cells above first."
    )
else:
    if MODE == "realtime":
        print(f"🔁 Entering REALTIME mode ({MODE}). Press Stop to cancel.")
        print("   (Polling every 60 seconds...)")
        while True:
            try:
                run_batch()
                print("💤 Sleeping 60s...")
                time.sleep(60)
            except KeyboardInterrupt:
                print("🛑 Stopped by user.")
                break
            except Exception as e:
                print(f"❌ Error in loop: {e}")
                time.sleep(60)
    else:
        print(f"▶️ Running SINGLE BATCH ({MODE}).")
        run_batch()

In [0]:
# 加载 bronze 表
df_bronze = spark.table("enterprise.bronze_ingestion.brz_crypto_ohlc_1m")

# 打印总行数
print(f"当前 Bronze 表中的数据总量: {df_bronze.count()} 行")

# 查看包含的源文件数量（这能告诉你读了多少个 jsonl.gz 文件）
file_count = df_bronze.select("source_file").distinct().count()
print(f"已处理的源文件数量: {file_count} 个")

In [0]:
%skip
# 检查前置条件
if "s3" not in globals() or "BUCKET" not in globals():
    print(
        "❌ 错误：S3 客户端未就绪。请先运行 Notebook 最上面的 'Imports' 和 'Configuration' 单元格！"
    )
else:
    print(f"✅ 环境检查通过。正在扫描 Bucket: {BUCKET} ...")

    # --- 1. 定义统计函数 (防止内存溢出的版本) ---
    def count_s3_files_only(bucket, prefix):
        count = 0
        paginator = s3.get_paginator("list_objects_v2")

        # 分页读取
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            if "Contents" in page:
                for obj in page["Contents"]:
                    if obj["Key"].endswith(".jsonl.gz"):
                        count += 1
                        # 每 500 个文件打印一次进度，避免让你觉得卡住了
                        if count % 500 == 0:
                            print(f"   已扫描 {count} 个文件...", end="\r")
        return count

    # --- 2. 执行 S3 统计 ---
    try:
        total_s3_files = count_s3_files_only(BUCKET, PREFIX)
        print(f"\n✅ S3 扫描完成！总共有: {total_s3_files} 个 .jsonl.gz 文件")
    except Exception as e:
        print(f"\n❌ S3 统计出错: {e}")
        total_s3_files = 0

    # --- 3. 获取 Bronze 表中的文件数 ---
    try:
        # 统计表中不重复的 source_file 数量
        df_bronze = spark.table("enterprise.bronze_ingestion.brz_crypto_ohlc_1m")
        processed_count = df_bronze.select("source_file").distinct().count()
        print(f"✅ Bronze 表已处理文件数: {processed_count}")
    except Exception as e:
        print(f"\n⚠️ 警告: 无法读取 Bronze 表 (可能表还不存在): {e}")
        processed_count = 0

    # --- 4. 打印最终对比结果 ---
    print("-" * 30)
    print("📊 总结报告:")
    print(f"   - S3 总文件数:     {total_s3_files}")
    print(f"   - Bronze 已读入:   {processed_count}")
    print(f"   - 剩余未读文件:    {total_s3_files - processed_count}")
    print("-" * 30)